# The mathematics a stage carries

<div class="alert alert-info">

**Summary**

This lesson constructs the consumption stage of the portfolio–consumption model and inspects what its object holds: its declarations as a signature, functions as terms, one law of motion in a forward and a backward model, the relation consumption satisfies, the recorded conditions and the encoding. The stage states its mathematics exactly enough for a machine to check it. The lesson evaluates two kernels and solves nothing.

</div>

<div class="alert alert-success">

**Relationship with other documents**

**Builds on** — [The stage as an object](../getting-started/01-the-stage.ipynb) and [Value functions and the backward builders](../getting-started/02-value-functions.ipynb): the stage, its builders, cards and kernels.

**Formal account** — [Appendix A — Semantics](../../docs/bellman-calculus/APPX-A-SEMANTICS.md) and [Appendix B — Initial-algebra semantics](../../docs/bellman-calculus/APPX-B-INITIAL-ALGEBRA.md) §§1 and 3: signatures, terms and their two interpretations.

**See also** — [API](../../docs/Bellman-Sym/API/index.md) §1 and [The stage](../../docs/Bellman-Sym/API/stage/01-the-stage.md) §5; Riehl (2016, Example 4.1.13 and Theorem 4.2.7), free categories; Goguen, Thatcher, Wagner and Wright (1977, §2), signatures.

</div>

<div class="alert">

**Key terminology**

| Term | Meaning |
|---|---|
| [[signature]] | Declared sets as types, and declared functions and laws with their input and output sets. Lesson 1's signature of a builder is here a builder signature. |
| model | An interpretation in the logician's sense, not the economic model: a set, its **carrier**, for each field and a map between carriers for each operation. |
| [[semantics#Denotation\|denotation]] | The mathematical object a model assigns to a piece of the stage. |
| mathematical type | A wire's [[sort]] and a term denoting its carrier. |
| kernel | The mathematical data an operation uses: a law of motion, a value aggregator. |
| obligation | A condition recorded with a status: `required` until something decides it, `established` with evidence. |
| addition | A `.bl` file beside the stage file, `methods.bl`, whose typing entries name a method at places of the stage and whose lines are read with the stage file's. |

</div>

The [[stage]] is that of the prerequisite lessons: resources $\mathrm m$, consumption $\mathrm c \in \Gamma(\mathrm m_d) = (0, \mathrm m_d]$, savings $\mathrm a = \mathrm m_d - \mathrm c$, utility $u(\mathrm c) = \mathrm c^{1-\rho}/(1-\rho)$. The file declares $\mathit{du}(\mathrm c) = \mathrm c^{-\rho}$, the derivative of $u$ by a calculation the package does not perform, states the maximization in its policy block, and states the derivative of the decision value by the line `D_{m_d}V = 𝑑𝑢(c)`. The addition `methods.bl` beside it, read in the section on the relation consumption satisfies, declares $\mathit{duinv}(x) = x^{-1/\rho}$, the inverse of $\mathit{du}$, and writes in the policy block the three lines of the method of endogenous gridpoints (Carroll 2006), the first-order conditions a maximizer satisfies at the resources $m_{d\succ}(\mathrm a)$ from which it saves $\mathrm a$, consuming $c_\succ(\mathrm a)$:

$$
c_\succ(\mathrm a) \le \mathit{duinv}\big(D_{\mathrm a}V_\succ(\mathrm a)\big) \perp \mathrm a \ge 0, \qquad
m_{d\succ}(\mathrm a) = \mathrm a + c_\succ(\mathrm a), \qquad
c\big(m_{d\succ}(\mathrm a)\big) = c_\succ(\mathrm a).
$$

The term `D_{a}V[>]` denotes the derivative of the continuation value in savings, $D_{\mathrm a}V_\succ$, whose values the next stage supplies with its value, by a derivative line or by the value's representation. With $\mathit{du} = u'$ and $\mathit{duinv} = (u')^{-1}$, `InvEuler` is the Euler equation: marginal utility is at least the marginal value of savings, with equality where $\mathrm a > 0$.

The lesson writes only the two files of the interactive view of the stage opened in the section on the forward and backward models, in the folder model-explorer beside it, and reads two files, the stage file and its addition:

- `applications/portfolio-choice/declaration/stages/cons/cons.bl`
- `applications/portfolio-choice/declaration/stages/cons/methods.bl`

The first code cell imports the set-up module, checkout, described in the first lesson, the standard module json, and the two functions called.

In [1]:
import checkout
import json
from bellman.stage import elaborate_stage, load_stage

## The declarations as a signature

`load_stage` reads the file and `elaborate_stage` constructs the stage, named `cons`.

In [2]:
cons = elaborate_stage(load_stage("applications/portfolio-choice/declaration/stages/cons/cons.bl"))

Construction compares the sets on which one block writes a [[field]] and another reads it. The text `variant` changes the header of `dcsn_to_arvl`, line 25, to read the decision value on the reals.

In [3]:
variant = load_stage("applications/portfolio-choice/declaration/stages/cons/cons.bl")["stage"]["text"].replace("(V) : !dcsn", "(V : ℝ @ !vlu) : !dcsn")

`elaborate_stage` takes a source record like those `load_stage` returns; this one holds the variant's text, no calibration, no settings and no addition, and is refused: the block `cntn_to_dcsn` writes the decision value on $[-\infty, \infty)$ and the variant's `dcsn_to_arvl` reads it on $\mathbb R$, and the first set does not lie in the second.

In [4]:
elaborate_stage({"stage": {"text": variant, "origin": "cons.bl"}, "calibration": {}, "settings": {}, "methods": []})

ConstructionError: cons.bl:20:1-23:2: the field V is produced by 'cntn_to_dcsn' on [-inf, inf) and consumed by 'dcsn_to_arvl' on ℝ, and [-inf, inf) does not lie in ℝ; declare the field with one set in both blocks, or state the control's feasible set in the maximization line (address 'cntn_to_dcsn'; rule set-mismatch)
  write: Declare V with one set in blocks 'cntn_to_dcsn' and 'dcsn_to_arvl', or state the control's feasible set in the maximization line

Line 20 writes $V$ on $[-\infty, \infty)$. The file's own line `V[<] = V`, line 26, can give $V_\prec$, declared on $\mathbb R$, the value $-\infty$: that is recorded as the condition $V_\prec(\mathrm m, \mathrm l) \in \mathbb R$ on the solved model, not refused.

The object holds the model in `definitions`, one record per declaration and block line, with laws of motion and value aggregators derived from the blocks, and in the graphs that `initial` and `builder` reach. `provenance` records the file read and the SHA-256 digest of its text; `metadata`, the `@` annotations besides the stage name, and `bindings`, the values, methods and settings bound, hold nothing. `list()` prints the table by kind.

In [5]:
cons.definitions.list()

BELLMAN-SYM / STAGE cons / DEFINITIONS                          [RECORDS = 17]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

SETS 2 ───────────────────────────────────────────────────────────────────────
  cons:set:Xm   Xm  = ℝ₊₊
  cons:set:Xa   Xa  = ℝ₊

PARAMETERS 1 ─────────────────────────────────────────────────────────────────
  cons:parameter:ρ   ρ  ∈ [1, ∞)

FUNCTIONS 2 ──────────────────────────────────────────────────────────────────
  cons:function:𝑢    𝑢  : ℝ₊₊ → ℝ,  c ↦ c^(1 − ρ)/(1 − ρ)
                     conditions ¬(1 − ρ = 0)
  cons:function:𝑑𝑢   𝑑𝑢  : ℝ₊₊ → ℝ₊₊,  c ↦ c^(−ρ)

CORRESPONDENCES 1 ────────────────────────────────────────────────────────────
  cons:correspondence:Γ   Γ  : ℝ₊₊ ⇉ ℝ₊₊,  x ↦ (0, x]

LAWS OF MOTION 2 ─────────────────────────────────────────────────────────────
  cons:g:arvl_to_dcsn   𝑔≺  : Xm × ℝ₊ → Xm,  (m, l) ↦ m
  cons:g:dcsn_to_cntn   𝑔≻  : {m_d, c ∈ Xm × ℝ₊₊ | c ∈ Γ(m_d)} → Xa,
                          

The table is the stage's [[signature]]: types `Xm` and `Xa`; symbols $u$, $\mathit{du}$, Γ, $g_\prec$ and $g_\succ$ with their input and output sets. The blocks add a graph whose vertices are the fields at their [[perch|perches]] and whose edges are operations: $\mathrm m \to \mathrm m_d$, block `arvl_to_dcsn`, and $\mathrm m_d \to \mathrm a$, block `dcsn_to_cntn`, the one decision edge, indexed by the consumption rule $c$. Paths of edges, joined end to end, are the arrows of the graph's **free category**; the builders graphs compose operations along them.

## A definition as a term

`denote(identity)` answers one record, displayed as its card.

In [6]:
cons.definitions.denote("cons:function:𝑢")

STAGE cons / FUNCTION 𝑢
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Defines
  𝑢  : ℝ₊₊ → ℝ,  c ↦ c^(1 − ρ)/(1 − ρ)

Conditions
  ¬(1 − ρ = 0)
──────────────────────────────────────────────────────────────────────────────
Source  cons.bl
  8  𝑢(c : ℝ₊₊) := c^(1-ρ)/(1-ρ)  :: [id=util]

Definitions referenced
  ρ     cons:parameter:ρ
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Line 8, `util`, denotes a function from $\mathbb R_{++}$ to the reals, provided the divisor $1 - \rho$ is nonzero, which $\rho \in [1, \infty)$ does not ensure. The record's `denoted` holds the term and type that state the denotation. Its `expression`, named `utility`, is the defining **term**: a tree of records, one per operation, number or name.

In [7]:
utility = cons.definitions.denote("cons:function:𝑢").denoted[0].expression

The term displays as the card's rule.

In [8]:
utility

c ↦ c^(1 − ρ)/(1 − ρ)

Its `body` is that tree, printed as the calls that build it.

In [9]:
utility.body

Div(left=Pow(base=Sym(symbol=Constant(name='cons:function:𝑢:parameter:0:c')), exponent=Sub(left=Number(value=1), right=Sym(symbol=Constant(name='cons:parameter:ρ')))), right=Sub(left=Number(value=1), right=Sym(symbol=Constant(name='cons:parameter:ρ'))))

A division, `Div`, whose numerator is a power, `Pow`, of $\mathrm c$ with exponent $1 - \rho$ and whose denominator is the subtraction, `Sub`, $1 - \rho$. Each name is an **identity**, the internal name of a declared thing. The term denotes $u$ exactly at every positive consumption; a numerical method's values at chosen points only approximate it.

Γ, line 10, is a term too; its body is an interval.

In [10]:
cons.definitions.denote("cons:correspondence:Γ").denoted[0].expression.body

Interval(bounds=Range(left=Open(), lower=Number(value=0), upper=Sym(symbol=Constant(name='cons:correspondence:Γ:parameter:0:x')), right=Closed()))

`Open` at 0 and `Closed` at $x$: consumption is positive and may exhaust resources.

## One law of motion by three routes

The law of motion of `dcsn_to_cntn` has its own record.

In [11]:
cons.definitions.denote("cons:g:dcsn_to_cntn")

STAGE cons / LAW OF MOTION dcsn_to_cntn
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Defines
  𝑔≻  : {m_d, c ∈ Xm × ℝ₊₊ | c ∈ Γ(m_d)} → Xa,  (m_d, c) ↦ m_d − c
──────────────────────────────────────────────────────────────────────────────
Source  cons.bl
  17  a = m_d - c

Definitions referenced
  Xm    cons:set:Xm
  Xa    cons:set:Xa
  Γ     cons:correspondence:Γ
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

It is $g_\succ(\mathrm m_d, \mathrm c) = \mathrm m_d - \mathrm c$ on the feasible pairs, with values in `Xa`, written on line 17. Its term is named `law`.

In [12]:
law = cons.definitions.denote("cons:g:dcsn_to_cntn").denoted[0].expression

The backward model gives the edge $\mathrm m_d \to \mathrm a$ the builder `cntn_to_dcsn`, whose first operation, the **pullback**, composes a function with the law; its record is named `pullback`.

In [13]:
pullback = cons.dcsn.builder.bwd["Pullback"].denote()

Its `expression` is the term for the map the operation denotes.

In [14]:
pullback.expression

V ↦ V ∘ 𝑔(·, c)

For each rule $c$ it maps a function $V$ of savings to $V \circ g_{\succ,c}$, a function of resources, where $g_{\succ,c}(\mathrm m_d) = g_\succ(\mathrm m_d, c(\mathrm m_d))$. The forward model gives the edge the builder `dcsn_to_cntn`, whose **push-forward** carries a distribution along the law.

In [15]:
pushforward = cons.dcsn.builder.fwd["Pushforward"].denote()

It maps distributions to distributions.

In [16]:
pushforward.expression

w ↦ measure on Xa given by B ↦ w({m_d ∈ Xm | 𝑔(m_d, c(m_d)) ∈ B})

Each set $B$ of savings receives the mass $w$ gives to the resources the rule sends into $B$. Each record's `kernel` holds the law it reads; `==` compares terms up to argument names, so equal terms denote one function.

In [17]:
pullback.kernel.denoted[0].expression == law, pushforward.kernel.denoted[0].expression == law

(True, True)

The identity each kernel cites shows one declaration, not two equal ones:

In [18]:
pullback.kernel.denoted[0].identity, pushforward.kernel.denoted[0].identity

('cons:g:dcsn_to_cntn', 'cons:g:dcsn_to_cntn')

The three routes reach one law, written once on line 17; an edit there changes all three. With $K(\mathrm m_d, \cdot)$ the point mass at $g_{\succ,c}(\mathrm m_d)$, the push-forward sends a distribution $\mu$ of resources to $\mu K$, the pullback a function $v$ of savings to $K^* v = v \circ g_{\succ,c}$. Where $\int |v| \,\mathrm d(\mu K)$ is finite, Fubini's theorem gives (Sargent and Stachurski 2026, Lemma A.5.33)

$$
\langle \mu K, v \rangle = \int v \,\mathrm d(\mu K) = \int K^* v \,\mathrm d\mu = \langle \mu, K^* v \rangle:
$$

the expected continuation value computed forward equals the one computed backward. Only these two operations pair; the aggregation's $u(\mathrm c)$ has no forward counterpart ([Operators and builders](../../docs/bellman-calculus/05-Operators%20and%20builders.md) §3). With a shock, the pullback followed by an expectation operation computes $K^* v$ ([the expectation over the same shocks](../getting-started/03-distributions.ipynb#The-expectation-over-the-same-shocks)).

## The forward and backward models

In a model, a field denotes its carrier and an edge its arrow: the forward model gives fields finite measures and edges push-forwards; the backward model, functions and backward builders. A path's arrow composes its edges' arrows, as in the forward builders graph:

In [19]:
cons.builder.fwd.denote()

STAGE cons / FORWARD BUILDERS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   μ≺  : finite measure on Xm × ℝ₊
       ────────────────────────────────────────
  out  μ≻  : finite measure on Xa

At
  𝑐  : Xm → ℝ₊₊,  𝑐(m_d) ∈ Γ(m_d)

Operations
     arvl_to_dcsn
     in: μ≺                             out: μ
     ──────────────────────────────────────────
     rule: μ : finite measure on Xm

  1    Pushforward
       in: μ≺                           out: μ
       ────────────────────────────────────────
       rule: μ(B) = μ≺({m≺, l≺ ∈ Xm × ℝ₊ | 𝑔≺(m≺, l≺) ∈ B})

     dcsn_to_cntn
     in: μ                              out: μ≻
     ──────────────────────────────────────────
     rule: μ≻ : finite measure on Xa

  2    Pushforward
       in: μ                            out: μ≻
       ────────────────────────────────────────
       rule: μ≻(B) = μ({m_d ∈ Xm | 𝑔≻(m_d, 𝑐(m_d)) ∈ B})

Conditions
  𝑢                         ¬(1 − ρ = 0)
  policy

Push-forward 1 carries $\mu_\prec$ to $\mu$ along $g_\prec$; push-forward 2 carries $\mu$ to $\mu_\succ$ along $g_{\succ,c}$. Backward, `cntn_to_dcsn` applies the aggregation after the pullback; [lesson 2](../getting-started/02-value-functions.ipynb) reads its kernel:

In [20]:
cons.dcsn.builder.bwd["Aggregation"].kernel()

𝐺 of cntn_to_dcsn: 𝐺(V≻, m_d, c) = 𝑢(c) + V≻
output V; unused m_d

At each point $G$ returns $u(\mathrm c) + V_\succ$; the derivative line's piece returns $\mathit{du}(\mathrm c)$ beside it. `denote()` and `kernel()` state what the models assign; the package computes no denotation.

`checkout.explore` shows the stage in [Google's Model Explorer](https://github.com/google-ai-edge/model-explorer), an interactive viewer in which each builder is a frame opened to the operations it holds, and in which selecting an operation shows, in a side panel, its kernel and the wires it reads and writes, each with its perch and its type; the kernel it shows for the aggregation of `cntn_to_dcsn` is the one displayed above. The view opens only when the lesson is run: the saved output of the next cell is the paths of the two files it writes into the folder `model-explorer` beside the lesson.

In [21]:
checkout.explore(cons)

tutorials/topics/model-explorer/cons.graph.json
tutorials/topics/model-explorer/cons.edge-overlays.json


`signature()` answers a **builder signature**, the wires an operation reads and writes. The decision builder's input wire, $V_\succ$, is named `wire`.

In [22]:
wire = cons.dcsn.builder.bwd.signature().inputs[0]

Its mathematical type pairs a sort with `carrier`, a term denoting the carrier. That term, and the forward model's for $\mathrm a$ from the push-forward's output wire:

In [23]:
(wire.mathematical_type.carrier, cons.dcsn.builder.fwd["Pushforward"].signature().outputs[0].mathematical_type.carrier)

(Xa → [−∞, ∞), finite measure on Xa)

They denote the measurable functions from `Xa` to $[-\infty, \infty)$ and the finite measures on `Xa`. A model gives $V$ one carrier, where the arrow of `cntn_to_dcsn` ends and that of `dcsn_to_arvl` begins; in the variant, the writing block's set $[-\infty, \infty)$ does not lie in the reading block's $\mathbb R$, so no one carrier serves both ends. The sort classifies the wire: kind; the field's perch, role and value set; coordinate; domain and result set.

In [24]:
wire.mathematical_type.sort

('function',
 ('cntn',
  'value',
  Interval(bounds=Range(left=Closed(), lower=Neg(operand=Sym(symbol=Constant(name='inf'))), upper=Sym(symbol=Constant(name='inf')), right=Open()))),
 ('cons:cntn:a',),
 Named(name='Xa'),
 Interval(bounds=Range(left=Closed(), lower=Neg(operand=Sym(symbol=Constant(name='inf'))), upper=Sym(symbol=Constant(name='inf')), right=Open())))

## The relation consumption satisfies

The policy builder of the stage file maximizes: its operations are the pullback of the continuation value along the law, the aggregation that forms the objective, the argmax over the feasible set, and the evaluation.

In [25]:
cons.dcsn.builder.policy.list()

1  Pullback                primitive
2  optimality:Aggregation  primitive
3  optimality:Argmax       primitive  → 𝑐
4  Evaluation              primitive  → 𝑐

The addition `methods.bl` beside the file writes the relation consumption satisfies, the three lines of the opening section, in a block that names the stage's block `policy`; read with the file, its lines stand in the builder in place of the maximization, which the declared builders, `declared`, keep. The next cell constructs the stage again from the file with the addition, which `load_stage` reads when its path is given, and names it `cons_egm`.

In [26]:
cons_egm = elaborate_stage(load_stage("applications/portfolio-choice/declaration/stages/cons/cons.bl", ["applications/portfolio-choice/declaration/stages/cons/methods.bl"]))

Its policy builder's operations follow the lines `InvEuler`, `grid` and `identification`.

In [27]:
cons_egm.dcsn.builder.policy.list()

1  InvEuler:Aggregation     primitive  → 𝑐≻
2  grid:Aggregation         primitive  → 𝑚_d≻
3  identification:Pullback  primitive  → 𝑐
4  Evaluation               primitive  → 𝑐

The first aggregation, `InvEuler:Aggregation`, holds the relation of the line `InvEuler` with its **clause** `⟂ a >= 0`, the condition of complementary slackness that joins the inequality to the borrowing limit $\mathrm a \ge 0$. Its builder signature writes its output, the **candidate** $c_\succ$, with a double arrow, `Xa ⇉ ℝ₊₊`: a **correspondence**, which gives at each level of savings the set of consumption levels the relation admits there, in $\mathbb R_{++}$, the set into which $\mathit{duinv}$ maps.

In [28]:
cons_egm.dcsn.builder.policy["InvEuler:Aggregation"].signature()

in         𝑉≻ : Xa → [−∞, ∞)
out        𝑐≻ : Xa ⇉ ℝ₊₊

Where $\mathrm a > 0$, the clause makes the relation an equality, $c_\succ(\mathrm a) = \mathit{duinv}(D_{\mathrm a}V_\succ(\mathrm a))$; at $\mathrm a = 0$, where the borrowing limit binds, it admits every consumption level up to $\mathit{duinv}(D_{\mathrm a}V_\succ(0))$, the point Carroll's equation (28) takes there ([Constraints](../../docs/Bellman-Sym/10-constraints.md) §4.3). The operation's kernel is the explicit side, $\mathit{duinv}(D_{\mathrm a}V_\succ)$, which `compile_kernel()` compiles ([Kernels as functions](kernels.ipynb)); the card keeps the relation. The second aggregation, `grid`, gives the resources $m_{d\succ}(\mathrm a) = \mathrm a + c_\succ(\mathrm a)$ from which each level of savings is reached, a correspondence as well. The third operation, `identification:Pullback`, the **identification**, equates the rule at those resources with the candidate, $c(m_{d\succ}(\mathrm a)) = c_\succ(\mathrm a)$; its output, the rule, is its own unknown, so it states a relation, and its signature answers the set of rules that satisfy it:

In [29]:
cons_egm.dcsn.builder.policy["identification:Pullback"].signature()

in         𝑐≻ : Xa ⇉ ℝ₊₊
in         𝑚_d≻ : Xa ⇉ Xm
out        {f ∈ {c ∈ (Xm → ℝ₊₊
           ) | c(m_d) ∈ Γ(m_d)} | ∀ a ∈ Xa, x ∈ 𝑐≻(a): (f(a + x) = x)} :
           𝒫({c ∈ (Xm → ℝ₊₊) | c(m_d) ∈ Γ(m_d)})

A set of functions: the feasible rules $f$ with $f(\mathrm a + x) = x$ at every level of savings $\mathrm a$ and every $x$ in $c_\succ(\mathrm a)$, the pairs the relation admits. The condition $c(\mathrm m_d) \in \Gamma(\mathrm m_d)$ writes $\mathrm m_d$ free; the package reads it at every $\mathrm m_d$. At zero savings the pairs are $(0, x)$ for every $x$ in $(0, \mathit{duinv}(D_{\mathrm a}V_\succ(0))]$, so every rule of the set consumes all resources at every $\mathrm m \le \mathit{duinv}(D_{\mathrm a}V_\succ(0))$, where the borrowing limit binds, as Carroll's equation (29) does: the clause on line 29 of the addition states this corner, and no method adds it. Where two admitted pairs give one level of resources two consumption levels, no rule satisfies both and the set is empty; when $m_{d\succ}$ increases, no two pairs do. The evaluation is the identity at $\mathrm m_d$:

In [30]:
cons_egm.dcsn.builder.policy["Evaluation"].kernel()

identity at m_d; no policy is inserted

It passes the set on, selecting no rule; the method `!egm`, which the addition types on the builder, selects one when the model is solved.

## The conditions the construction records

`obligations()` lists each obligation, a status other than `required` in brackets; the next cell lists those of `cons`, the stage constructed from its file alone.

In [31]:
cons.obligations()

arithmetic-definedness      ¬(1 − ρ = 0)
finite-real-arithmetic      ¬(𝑢(cp) ∈ [∞, ∞] ∧ V[>] ∈ [−∞, −∞])
aggregation-admissibility   V(m_d) ∈ [−∞, ∞)
marginal-reading            the right side of each derivative line equals the derivative, in the named coordinate, of the value the builder writes, under the envelope conditions: concavity of the continuation value, interiority of the maximizer or the multiplier's term at a binding constraint, differentiability of the reward in the control, a measurable selection of the policy, and differentiability of the value at the point read; listed and not checked
aggregation-admissibility   V≺(m) ∈ ℝ
range-compatibility         m_d − c(m_d) ∈ Xa   [established]
range-compatibility         V≺(m, l) ∈ ℝ
continuation-input          The representing function for V[>] must be measurable on its declared domain, depend only on (a) in that order, and take values in [-inf, inf).

| Condition | Line | Decided by | If it fails |
|---|---|---|---|
| `arithmetic-definedness`, $1 - \rho \ne 0$ | `util` | calibration | calibration refuses |
| `finite-real-arithmetic`, not both $u(\mathrm{cp}) = \infty$ and $V_\succ = -\infty$ | `optimality` | the first row: $u(\mathrm{cp})$ is finite where $1 - \rho \ne 0$ | cannot fail where the first row holds |
| `aggregation-admissibility`, $V(\mathrm m_d) \in [-\infty, \infty)$ | `value` | the declared sets | cannot fail |
| `marginal-reading` | `marginal` | the envelope conditions, assumed | $\mathit{du}(\mathrm c)$ is not the derivative of $V$ in $\mathrm m_d$ |
| `aggregation-admissibility`, $V_\prec(\mathrm m) \in \mathbb R$ | `value` of `dcsn_to_arvl` | the solved value $V$ | $V_\prec = -\infty$ |
| `range-compatibility`, $\mathrm m_d - c(\mathrm m_d) \in$ `Xa` | `dcsn_to_cntn.1` (line 17) | the declarations, at construction | cannot fail |
| `range-compatibility`, $V_\prec(\mathrm m, \mathrm l) \in \mathbb R$ | `value` of `dcsn_to_arvl` | the solved value $V$ | $V_\prec = -\infty$ |
| `continuation-input` | the input `V[>]` | the function supplied for $V_\succ$ | the function is not measurable, does not depend on $\mathrm a$ alone, or takes a value outside $[-\infty, \infty)$ |

The second and third rows follow by algebra the package does not perform: $u(\mathrm{cp})$ is finite at every $\mathrm{cp} > 0$ where $1 - \rho \ne 0$, and $V_\succ < \infty$ gives $V < \infty$. The maximization the line `optimality` states is recorded as the one condition on its sum and no other. The stage read with the addition, `cons_egm`, records four conditions more, those of the addition's lines: that $m_{d\succ}(\mathrm a)$ lies in `Xm`, which follows from $\mathrm a \in \mathbb R_+$ and $c_\succ \in \mathbb R_{++}$; that the argument passed to $\mathit{duinv}$, the derivative of the continuation value, is positive; `evaluation-domain`, under which the representation that the addition types `!Linear` is evaluated inside the interval its nodes span or through an identified extension; and `egm-eligibility`, under which a rule of the identification's set maximizes the declared objective: $V_\succ$ concave and continuously differentiable with derivative $D_{\mathrm a}V_\succ$, with $\mathit{du} = u'$ and $\mathit{duinv} = (u')^{-1}$ ([Appendix C](../../docs/Bellman-Sym/appendices/14-appendix-c-syntax-to-semantics.md#45-an-implicit-statement-with-a-constrained-policy) §4.5), none of which the package checks. The corner needs no such assumption, since the clause states it: the set holds no rule that consumes less than all resources at some $\mathrm m \le \mathit{duinv}(D_{\mathrm a}V_\succ(0))$, such as $f(\mathrm m) = \mathrm m/2$ there, which is no maximum. The evidence of the established `range-compatibility` condition:

In [32]:
cons.obligations()[5].evidence

('declared choice set for cons:dcsn:c: (0, m_d]',
 'arithmetic range inclusion at dcsn_to_cntn.1')

The choice set $(0, \mathrm m_d]$ gives $\mathrm m_d - \mathrm c \ge 0$. `calibrate` ([lesson 4](../getting-started/04-parameters-methods-and-settings.ipynb)) decides the first condition, refusing a value that makes a condition on parameters alone false.

In [33]:
cons.calibrate({"ρ": 1.0})

ValueError: stage 'cons', cons:function:𝑢: calibration ρ = 1.0 makes ¬(1 - ρ = 0) false; calibrate requires every parameter-only obligation to hold (AMD-SG-022 §4); write parameter values satisfying ¬(1 - ρ = 0) in the calibration source

$\rho = 2$ gives numbers checkable by hand; the version calibrated there is `at_rho_2`.

In [34]:
at_rho_2 = cons.calibrate({"ρ": 2.0})

The divisor condition is now established, the value its evidence; six stay `required`.

In [35]:
at_rho_2.obligations()[0].evidence

('calibration ρ = 2.0',)

## The terms evaluated

`compile_kernel()` ([lesson 4](../getting-started/04-parameters-methods-and-settings.ipynb)) turns a kernel into a Python function built from the kernel's source text, which the lesson [Kernels as functions](kernels.ipynb) prints. The call in the next cell passes the version's value of $\rho$, which `kernel()` holds under `parameter_values`, after the kernel's own arguments. No public call evaluates $u$ alone, so $G$ is read at $V_\succ = 0$, $\mathrm m_d = 2$, $\mathrm c = 0.5$: $u(0.5) = 0.5^{-1}/(-1) = -2$.

In [36]:
print(at_rho_2.dcsn.builder.bwd["Aggregation"].compile_kernel()(0.0, 2.0, 0.5, *at_rho_2.dcsn.builder.bwd["Aggregation"].kernel().parameter_values))

-2.0


The push-forward's kernel is the law: $g_\succ(2, 0.5) = 1.5$.

In [37]:
print(at_rho_2.dcsn.builder.fwd["Pushforward"].compile_kernel()(2.0, 0.5))

1.5


## The encoding

`encoding()` writes the whole model as dictionaries and lists, a document another program reads; `derived` holds the answers of `denote()`, which state the denotations.

In [38]:
encoding = cons.encoding()

Written as strict JSON text, with no infinite number, and read back, it is unchanged:

In [39]:
json.loads(json.dumps(encoding, allow_nan=False)) == encoding

True

Its **schema** names the package's JSON Schema, which describes the whole document.

In [40]:
encoding["schema"]

'https://econ-ark.org/bellman-sym/stage-encoding/2.6'

Entry 3 of the derived definitions, counted from 0 in the table's order, is $u$'s record; the encoding writes a function's body as a list of one entry, here the utility term:

In [41]:
encoding["derived"]["definitions"][3]["value"]["fields"]["denoted"][0]["fields"]["expression"]["body"][0]

{'node': 'terms.Div',
 'fields': {'left': {'node': 'terms.Pow',
   'fields': {'base': {'node': 'terms.Sym',
     'fields': {'symbol': {'node': 'terms.Constant',
       'fields': {'name': {'ref': 'bound', 'depth': 0, 'position': 0}}}}},
    'exponent': {'node': 'terms.Sub',
     'fields': {'left': {'node': 'terms.Number', 'fields': {'value': 1}},
      'right': {'node': 'terms.Sym',
       'fields': {'symbol': {'node': 'terms.Constant',
         'fields': {'name': {'ref': 'document',
           'id': 'cons:parameter:ρ'}}}}}}}}},
  'right': {'node': 'terms.Sub',
   'fields': {'left': {'node': 'terms.Number', 'fields': {'value': 1}},
    'right': {'node': 'terms.Sym',
     'fields': {'symbol': {'node': 'terms.Constant',
       'fields': {'name': {'ref': 'document', 'id': 'cons:parameter:ρ'}}}}}}}}}

$\mathrm c$ is `{'ref': 'bound', 'depth': 0, 'position': 0}`, the first argument of the nearest enclosing function, so terms differing only in argument names are written alike; $\rho$ refers to `cons:parameter:ρ`. A checker outside the package reads here the mathematics this lesson displayed.

## What was checked and what is assumed

Checked by inspection: one law reached by three routes under one identity; two carriers for $\mathrm a$; a wiring the declared sets do not admit, refused; the divisor condition's evidence at $\rho = 2$, and $u$ and $g_\succ$ evaluated there; the encoding unchanged through JSON. Assumed of the model: the six `required` conditions of the stage file, two following by algebra the package does not perform, and the four the addition's lines add; and $\mathit{du} = u'$, $\mathit{duinv} = (u')^{-1}$, $V_\succ$ concave, and the derivative the next stage supplies equal to $V_\succ'$, under which the rule the identification states is the maximizer. The corner, a rule consuming all resources at every $\mathrm m \le \mathit{duinv}(D_{\mathrm a}V_\succ(0))$, is assumed nowhere: the clause on line 29 of the addition states it.